In [4]:
##### Calculates final capital and labor intensities using final production and capital/labor rasters (after re-scaling)

import os
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from pyproj import Transformer
from pathlib import Path

In [5]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import data
capital = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD.tif")
capital_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD_p10.tif")
capital_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_capital_USD_p90.tif")

labor = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs.tif")
labor_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs_p10.tif")
labor_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/rescaled_jobs_p90.tif")

production = rio.open_rasterio(f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif")

In [6]:
##### Calculate and save intensity rasters (central, p10, p90)

def compute_and_save_intensity(numerator, production_masked, crs, scale, out_path):

    intensity = (numerator / production_masked) * scale
    intensity = intensity.where(np.isfinite(intensity))
    intensity = intensity.rio.write_nodata(np.nan)
    intensity = intensity.rio.write_crs(crs)

    intensity.rio.to_raster(out_path, dtype="float32", compress="LZW")
    return intensity


# set CRS
crs = capital.rio.crs

# mask production once
production_masked = production.where(production >= 0.1)

out_dir = f"{cd}/Results/Raster_model"

variants = {
    "": {"capital": capital, "labor": labor},
    "_p10": {"capital": capital_p10, "labor": labor_p10},
    "_p90": {"capital": capital_p90, "labor": labor_p90},
}

results = {}

for suffix, data in variants.items():
    results[f"capital{suffix}"] = compute_and_save_intensity(
        numerator=data["capital"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/capital_intensity_USD_per_tonne{suffix}.tif",
    )

    results[f"labor{suffix}"] = compute_and_save_intensity(
        numerator=data["labor"],
        production_masked=production_masked,
        crs=crs,
        scale=1e3,
        out_path=f"{out_dir}/labor_intensity_thousand_jobs_per_tonne{suffix}.tif",
    )